# Lab 0 Task 0.1 — CIFAR-10 CNN Experiments
This notebook implements a simple CNN for CIFAR-10 classification and compares different optimizers and activation functions as required for Task 0.1 of Lab 0.

## Setup & Imports

In [1]:
!nvidia-smi # Take a look at the GPU

Wed Mar 25 19:43:47 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2080 Ti     On  |   00000000:1A:00.0 Off |                  N/A |
| 29%   25C    P8              1W /  250W |       1MiB /  11264MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%pip install numpy
%pip install torch torchvision
%pip install wandb

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [3]:
import warnings
import wandb
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from torchvision.datasets import CIFAR10
from typing import List, Dict, Tuple, Any

# Suppress NumPy 2.4 VisibleDeprecationWarning triggered inside torchvision
warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

2.11.0+cu130
True
NVIDIA GeForce RTX 2080 Ti
Using device: cuda


## Data Loading

In [4]:
BATCH_SIZE = 256 # Same batch size throughout notebook

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

train_set = CIFAR10(root="../data", train=True,  download=True, transform=transform)
test_set  = CIFAR10(root="../data", train=False, download=True, transform=transform)

classes: list[str] = train_set.classes
print("Classes:", classes)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=8, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True)

Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']


## Model Definition

The `activation` argument lets us swap between `LeakyReLU` and `Tanh` without rewriting the class.

In [5]:
class SimpleCNN(nn.Module):
    """
    A simple CNN for CIFAR-10 classification.

    Architecture:
        Conv Block 1 : Conv2d(3  → 32)  → act → MaxPool2d  (32×32 → 16×16)
        Conv Block 2 : Conv2d(32 → 64)  → act → MaxPool2d  (16×16 →  8×8)
        Conv Block 3 : Conv2d(64 → 128) → act → MaxPool2d  ( 8×8  →  4×4)
        FC 1         : Linear(128*4*4 → 256) → act
        FC 2         : Linear(256 → num_classes)
    """

    def __init__(self, num_classes: int = 10, activation: nn.Module = nn.LeakyReLU()) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(3,  32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)

        self.act  = activation
        self.pool = nn.MaxPool2d(2, 2)

        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.pool(self.act(self.conv1(x)))
        x = self.pool(self.act(self.conv2(x)))
        x = self.pool(self.act(self.conv3(x)))
        x = torch.flatten(x, start_dim=1)
        x = self.act(self.fc1(x))
        return self.fc2(x)

## Training & Evaluation Helpers

`train_one_epoch` and `validate` each handle a single pass through their respective loader.  
`fit` orchestrates the loop, tracks history, and restores the best checkpoint.

In [6]:
def train(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
) -> Tuple[float, float]:
    """Run one full pass over `loader` in training mode."""
    model.train()
    running_loss = 0.0
    correct      = 0
    total        = 0

    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        correct      += (outputs.argmax(dim=1) == labels).sum().item()
        total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def validate(
    model: nn.Module,
    loader: DataLoader,
) -> Tuple[float, float]:
    """Run one full pass over `loader` in evaluation mode."""
    model.eval()
    running_loss = 0.0
    correct      = 0
    total        = 0

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            loss    = criterion(outputs, labels)

            running_loss += loss.item() * inputs.size(0)
            correct      += (outputs.argmax(dim=1) == labels).sum().item()
            total        += labels.size(0)

    return running_loss / total, 100.0 * correct / total


def fit(
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    val_loader: DataLoader,
    wandb_kwargs: Dict[str, Any],
    num_epochs: int = 10,
) -> Dict[str, List[float]]:
    """Train `model` for `num_epochs`, validating after every epoch.

    Initialises and closes a wandb run for the duration of training.
    Saves the weights that achieved the best validation accuracy and
    restores them at the end.

    Returns
    -------
    history : dict with keys 'train_loss', 'val_loss', 'train_acc', 'val_acc'
    """

    with wandb.init(**wandb_kwargs):

        best_val_acc = 0.0
        best_state   = None
        history: dict[str, list[float]] = {
            "train_loss": [], "val_loss": [],
            "train_acc":  [], "val_acc":  [],
        }
    
        for epoch in range(1, num_epochs + 1):
            train_loss, train_acc = train(model, train_loader, optimizer)
            val_loss,   val_acc   = validate(model, val_loader)
    
            history["train_loss"].append(train_loss)
            history["val_loss"].append(val_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)
    
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                best_state   = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    
            print(
                f"Epoch {epoch:>{len(str(num_epochs))}}/{num_epochs} | "
                f"train loss {train_loss:.4f}, train acc {train_acc:.2f}% | "
                f"val loss {val_loss:.4f}, val acc {val_acc:.2f}%"
            )
    
            wandb.log({
                "Training Loss": train_loss,
                "Training Accuracy": train_acc,
                "Validation Loss": val_loss,
                "Validation Accuracy": val_acc
            })
    
    # Restore best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"\nRestored best weights (val acc {best_val_acc:.2f}%)")

    return history

## Experiment A – SGD + LeakyReLU
*(The original baseline from the script)*

In [7]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-2

criterion = nn.CrossEntropyLoss()

model_a = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_a = optim.SGD(model_a.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="SGD + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "SGD", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_a = fit(model_a, optimizer_a, train_loader, test_loader,
                wandb_kwargs=wandb_kwargs, num_epochs=NUM_EPOCHS)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: oscar-engelmark (d7047e-group12) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch  1/50 | train loss 2.2985, train acc 13.75% | val loss 2.2930, val acc 15.98%
Epoch  2/50 | train loss 2.2842, train acc 18.54% | val loss 2.2700, val acc 22.88%
Epoch  3/50 | train loss 2.2348, train acc 23.46% | val loss 2.1734, val acc 24.59%
Epoch  4/50 | train loss 2.1053, train acc 25.76% | val loss 2.0509, val acc 26.94%
Epoch  5/50 | train loss 2.0159, train acc 28.78% | val loss 1.9684, val acc 30.98%
Epoch  6/50 | train loss 1.9244, train acc 32.08% | val loss 1.8802, val acc 33.43%
Epoch  7/50 | train loss 1.8283, train acc 35.01% | val loss 1.8142, val acc 33.93%
Epoch  8/50 | train loss 1.7623, train acc 36.97% | val loss 1.7154, val acc 38.76%
Epoch  9/50 | train loss 1.7064, train acc 38.74% | val loss 1.7292, val acc 38.09%
Epoch 10/50 | train loss 1.6566, train acc 40.45% | val loss 1.6869, val acc 38.28%
Epoch 11/50 | train loss 1.6140, train acc 41.87% | val loss 1.6761, val acc 39.77%
Epoch 12/50 | train loss 1.5759, train acc 43.42% | val loss 1.5961, val acc

Training Accuracy,▁▂▂▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇███████
Training Loss,███▇▇▆▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁
Validation Accuracy,▁▂▂▃▃▄▄▄▄▅▅▆▆▅▆▆▆▆▆▆▇▇▆▇▇▇▇▇▇▇▇▇█▇██████
Validation Loss,█▇▇▆▆▅▅▅▅▄▄▄▃▄▃▃▃▃▃▃▃▂▃▂▂▃▂▂▂▂▂▂▁▂▁▁▁▁▁▁
Training Accuracy,69.042
Training Loss,0.89148
Validation Accuracy,64.18
Validation Loss,1.03415



Restored best weights (val acc 64.25%)


## Experiment B – Adam + LeakyReLU
*(Task: swap SGD for Adam and report accuracy)*

In [8]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_b     = SimpleCNN(num_classes=len(classes), activation=nn.LeakyReLU()).to(device)
optimizer_b = optim.Adam(model_b.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + LeakyReLU",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "LeakyReLU",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_b = fit(model_b, optimizer_b, train_loader, test_loader,
                wandb_kwargs=wandb_kwargs, num_epochs=NUM_EPOCHS)

Epoch  1/50 | train loss 1.5586, train acc 43.23% | val loss 1.3156, val acc 52.16%
Epoch  2/50 | train loss 1.1682, train acc 58.39% | val loss 1.0932, val acc 60.86%
Epoch  3/50 | train loss 0.9684, train acc 66.04% | val loss 0.9471, val acc 66.40%
Epoch  4/50 | train loss 0.8347, train acc 70.71% | val loss 0.8505, val acc 69.96%
Epoch  5/50 | train loss 0.7316, train acc 74.51% | val loss 0.8269, val acc 71.60%
Epoch  6/50 | train loss 0.6472, train acc 77.54% | val loss 0.7770, val acc 72.90%
Epoch  7/50 | train loss 0.5652, train acc 80.21% | val loss 0.7463, val acc 74.46%
Epoch  8/50 | train loss 0.4985, train acc 82.45% | val loss 0.8097, val acc 73.77%
Epoch  9/50 | train loss 0.4250, train acc 85.06% | val loss 0.7617, val acc 75.03%
Epoch 10/50 | train loss 0.3585, train acc 87.66% | val loss 0.7869, val acc 75.79%
Epoch 11/50 | train loss 0.2939, train acc 89.78% | val loss 0.8112, val acc 76.01%
Epoch 12/50 | train loss 0.2314, train acc 92.08% | val loss 0.8676, val acc

Training Accuracy,▁▃▄▄▅▆▆▆▇▇▇▇████████████████████████████
Training Loss,█▆▅▅▄▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▄▅▆▇▇▇█████████████████▇███▇▇█████▇████
Validation Loss,▄▃▂▁▁▁▁▁▁▁▂▂▂▃▃▄▄▅▅▅▆▆▅▆▇▆▆▆▇▇▆▇▆▇▇▇█▇▇▇
Training Accuracy,99.232
Training Loss,0.02348
Validation Accuracy,75.67
Validation Loss,2.11538



Restored best weights (val acc 76.21%)


## Experiment C – Adam + Tanh
*(Task: swap LeakyReLU for Tanh and report accuracy)*

In [9]:
# Hyperparameters
NUM_EPOCHS = 50
LEARNING_RATE = 1e-3

model_c     = SimpleCNN(num_classes=len(classes), activation=nn.Tanh()).to(device)
optimizer_c = optim.Adam(model_c.parameters(), lr=LEARNING_RATE)

wandb_kwargs = dict(
    entity="d7047e-group12",
    project="Lab0",
    name="Adam + Tanh",
    tags=["Task 0.1"],
    config={"optimizer": "Adam", "lr": LEARNING_RATE, "activation": "Tanh",
            "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE},
)

history_c = fit(model_c, optimizer_c, train_loader, test_loader,
                wandb_kwargs=wandb_kwargs, num_epochs=NUM_EPOCHS)

Epoch  1/50 | train loss 1.4310, train acc 48.56% | val loss 1.1831, val acc 57.86%
Epoch  2/50 | train loss 1.0369, train acc 63.43% | val loss 1.0068, val acc 64.10%
Epoch  3/50 | train loss 0.8620, train acc 69.89% | val loss 0.8897, val acc 68.87%
Epoch  4/50 | train loss 0.7407, train acc 74.34% | val loss 0.8542, val acc 70.66%
Epoch  5/50 | train loss 0.6390, train acc 78.03% | val loss 0.8014, val acc 72.39%
Epoch  6/50 | train loss 0.5439, train acc 81.53% | val loss 0.7733, val acc 73.49%
Epoch  7/50 | train loss 0.4576, train acc 84.67% | val loss 0.7638, val acc 74.60%
Epoch  8/50 | train loss 0.3681, train acc 87.97% | val loss 0.8028, val acc 73.69%
Epoch  9/50 | train loss 0.2877, train acc 91.10% | val loss 0.8189, val acc 73.88%
Epoch 10/50 | train loss 0.2232, train acc 93.64% | val loss 0.8636, val acc 73.61%
Epoch 11/50 | train loss 0.1575, train acc 96.16% | val loss 0.8786, val acc 73.65%
Epoch 12/50 | train loss 0.1058, train acc 98.00% | val loss 0.8741, val acc

Training Accuracy,▁▃▄▅▅▆▇▇████████████████████████████████
Training Loss,█▆▅▅▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Accuracy,▁▃▅▆▇▇▇▇████████████████████████████████
Validation Loss,▅▃▂▂▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
Training Accuracy,100
Training Loss,0.00012
Validation Accuracy,75.43
Validation Loss,1.50209



Restored best weights (val acc 75.70%)
